# Data Formatting

The scripts here convert datasets into the formats used for the dataanalysis

## Events Manual Labels

Convert events encodings from CSV to NC

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

events_coding_files = ["waddendrifters2023_grounding_events_manual_coding","waddendrifters2023_grounding_events_manual_coding_validation"]

for ec_file in events_coding_files:
  df = pd.read_csv("data/in/%s.csv"%ec_file, engine="c",header=5)
  indices = np.array([[int(n) for n in ts_name.split('_')] for ts_name in df['ts_name']])
  df['irecord'] = indices[:,0]
  df['its'] = indices[:,1]
  df = df.drop(columns=['ts_name', 'comment'])
  ds = xr.Dataset.from_dataframe(df)
  sections = np.unique([[int(ir),int(its)] for ir in ds.irecord for its in ds.its],axis=0)
  ds.attrs = {"occurring_irecord" : sections[:,0], "occurring_its" : sections[:,1]}
  ds.to_netcdf('data/out/%s.nc'%ec_file)
  print("written data/out/%s.nc"%ec_file)

ds

## Format bathymetry dataset 'H1_Bathymetrie_2019'

Input: Download of complete area in format 'text/plain' from https://viewer.openearth.nl/wadden-viewer/download/geoserver?layers=96741014

In [ ]:
import numpy as np
import netCDF4
import matplotlib.pyplot as plt

in_txt_file_path = 'data/in/H1__Bathymetry__2019__.txt'
no_line_data_start = 20
abs_value_nan_threshold = 1e10
#lines are along longitude
grid_bounds = [(4.381381291349735, 52.79103152363201), (7.453523670336998, 53.73973667942354)]
out_nc_file_path = 'data/out/H1_Bathymetry_2019.nc'

print("IMPORTANT: verify parameters (eg grid boundaries) defined at the beginning of the input txt file")
print(grid_bounds)

# parse data
ftxt=open(in_txt_file_path)
lines=ftxt.readlines()
bath = np.array([[float(value) for value in line.strip().split()] for line in lines[no_line_data_start:]])
bath = np.flip(bath.T, axis=1)
bath[np.abs(bath)>abs_value_nan_threshold] = float('nan')
dat_depth = bath
n_lon, n_lat = dat_depth.shape
dat_lon = np.linspace(grid_bounds[0][0], grid_bounds[1][0], num=n_lon)
dat_lat = np.linspace(grid_bounds[0][1], grid_bounds[1][1], num=n_lat)

# write data
file_netcdf = netCDF4.Dataset(out_nc_file_path, 'w', format='NETCDF4')
file_netcdf.description = 'Bathymetry of Wadden sea area H1. Bathymetrie (2019) https://datahuiswadden.openearth.nl/geonetwork/srv/dut/catalog.search#/metadata/96741014'
file_netcdf.createDimension('lon', n_lon)
file_netcdf.createDimension('lat', n_lat)
var_lon = file_netcdf.createVariable('lon', 'f4', ('lon',))
var_lat = file_netcdf.createVariable('lat', 'f4', ('lat',))
var_depth = file_netcdf.createVariable('depth', 'f4', ('lon', 'lat',))
var_lon.units = 'degrees east'
var_lat.units = 'degrees north'
var_depth.units = 'meters compared to NAP'
var_lon[:] = dat_lon
var_lat[:] = dat_lat
var_depth[:] = dat_depth
file_netcdf.close()

# output data
file_netcdf = netCDF4.Dataset(out_nc_file_path, format='NETCDF4')
plt.imshow(dat_depth, origin ='lower')
plt.show()
print(file_netcdf)
file_netcdf.close()

## Optional: Format dataset 'swan_kuststrook_harmonie'

Input filenames

maps2d_swan_kuststrook_harmonie_202311140000.nc ; ... ; maps2d_swan_kuststrook_harmonie_202312040000.nc

In [ ]:
output_file = "data/out/waterlevel_NAP_from_Nov14_to_Dec05_2023_MATROOS_maps2d_swan_kuststrook_harmonie"

import netCDF4
import numpy as np
import json
import xarray as xr
import pandas as pd
import numpy as np

# open first file 2023-Nov-14 for info
if True:
  print(xr.open_dataset("data/in/maps2d_swan_kuststrook_harmonie_202311140000.nc", engine="netcdf4"))

MMDDs = ["1114", "1115", "1116", "1117", "1118", "1119", "1120", "1121", "1122", "1123", "1124", "1125", "1126", "1127", "1128", "1129", "1130", "1201", "1202", "1203", "1204"]

waterlevel = {'time' : [], 'set' : []}
for mmdd in MMDDs:
  print(mmdd)
  data = netCDF4.Dataset('data/in/maps2d_swan_kuststrook_harmonie_2023'+mmdd+'0000.nc', format='NETCDF4')

  nt = len(data['time'][:])
  nr = len(data['row'][:])
  nc = len(data['col'][:])

  data_t = data['time'][:]*60
  data_wl = data['sep'][:].filled(fill_value=np.nan)
  data_lon = data['lon'][:].filled(fill_value=np.nan)
  data_lat = data['lat'][:].filled(fill_value=np.nan)

  # count number of valid positions
  npositions = 0
  for ir in range(nr):
    for ic in range(nc):
      if np.isnan(data_lon[ir,ic]) or np.isnan(data_lat[ir,ic]):
        continue
      npositions += 1

  for it in range(nt):
    waterlevel['time'] += [data_t[it]]
    dat_lon = np.zeros(npositions)
    dat_lat = np.zeros(npositions)
    dat_level = np.zeros(npositions)
    ind = 0
    for ir in range(nr):
      for ic in range(nc):
        if np.isnan(data_lon[ir,ic]) or np.isnan(data_lat[ir,ic]):
          continue
        dat_level[ind] = data_wl[it,ir,ic]
        dat_lon[ind] = data_lon[ir,ic]
        dat_lat[ind] = data_lat[ir,ic]
        ind += 1
    data_set = {'lon' : dat_lon, 'lat' : dat_lat, 'level' : dat_level}
    waterlevel['set'] += [data_set]

np.save(output_file, waterlevel)

## Optional: Format dataset 'waterinfo_rws'

Download at https://waterinfo.rws.nl/publiek/waterhoogte/ under 'Download historische data'

Time period 14-11-2023 to 01-12-2023

Stations selection is flexible, e.g. {Den Helder Marsdiep, Harlingen Waddenzee, Ameland Nes, Vlieland haven}

In [ ]:
file_name_input_csv = "rws_waterlevels_14Nov_1Dec.csv"
file_name_output_nc = file_name_input_csv[:-3] + "nc"

import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

def epoch_timestamp_from_timestringYMDHMSZ(timestringYMDHMSZ):
  sformat = '%d-%m-%YT%H:%M:%S %Z'
  t1 = datetime.strptime(timestringYMDHMSZ, sformat)
  t0 = datetime.strptime('01-01-1970T00:00:00 %s'%timestringYMDHMSZ[20:], sformat)
  return (t1-t0).total_seconds()

table = pd.read_csv("data/in/"+file_name_input_csv, delimiter=';')

n_rows = len(table)
table_timestamps = np.array([epoch_timestamp_from_timestringYMDHMSZ("%sT%s CET" % (table.WAARNEMINGDATUM[i], table.WAARNEMINGTIJD[i])) for i in range(n_rows)])
times = np.unique(table_timestamps)
n_times = len(times)
location_codes = np.unique(table.LOCATIE_CODE)
n_locs = len(location_codes)
locs = np.zeros((n_locs,2)) #(lon,lat)
for i in range(n_locs):
  iloc = np.argwhere(table.LOCATIE_CODE==location_codes[i])[0]
  iloc = iloc if isinstance(iloc, int) else iloc[0]
  locs[i,0], locs[i,1] = table.LON[iloc], table.LAT[iloc]
waterlevels = np.zeros((n_times,n_locs))
for it in range(n_times):
  for il in range(n_locs):
    ind = np.argwhere((table_timestamps==times[it])&(table.LOCATIE_CODE==location_codes[il]))[0]
    ind = ind if isinstance(ind, int) else ind[0]
    waterlevels[it,il] = table.NUMERIEKEWAARDE.values[ind]/100.

dat = xr.Dataset(
  data_vars = {
    "timestamp": (["itime"],times),
    "lon": (["ipos"],locs[:,0]),
    "lat": (["ipos"],locs[:,1]),
    "waterlevel":  (["itime","ipos"],waterlevels),
  },
  coords={
    "itime": np.array(range(n_times)),
    "ipos": np.array(range(n_locs))
  })
dat.attrs = {"units" : "s, m (NAP), deg"}
dat.to_netcdf('data/out/'+file_name_output_nc)

dat